# 02 — Feature Engineering

**Goal:** Derive new features that capture risk signals not directly present in the raw data, then validate them.

**New features:**
| Feature | Rationale |
|---|---|
| `amount_log` | Right-skewed amount → log-normalise for LR |
| `amount_x_velocity` | High spend + high velocity = compounded risk |
| `high_risk_hour` | Night hours (22:00–06:00) historically higher fraud |
| `velocity_bin` | Ordinal quartile bin for tree models |

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data_loader import load_raw, load_engineered, TARGET

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

## 1. Raw vs engineered dataset

In [ ]:
df_raw = load_raw()
df_eng = load_engineered()

new_cols = [c for c in df_eng.columns if c not in df_raw.columns]
print(f'Original features : {df_raw.shape[1]}')
print(f'After engineering : {df_eng.shape[1]}')
print(f'New features      : {new_cols}')

df_eng[new_cols + [TARGET]].head()

## 2. Validate amount_log

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, col, title in zip(axes,
                          ['amount', 'amount_log'],
                          ['Raw amount (right-skewed)', 'Log-transformed amount']):
    for label, color in [(0, 'steelblue'), (1, 'tomato')]:
        ax.hist(df_eng[df_eng[TARGET] == label][col], bins=50, alpha=0.6,
                color=color, label='Legit' if label == 0 else 'Fraud', density=True)
    ax.set_title(title)
    ax.legend()

plt.suptitle('Effect of log-transforming transaction amount', fontsize=12)
plt.tight_layout()
plt.savefig('../reports/figures/02_amount_log_transform.png', bbox_inches='tight')
plt.show()

## 3. Validate amount_x_velocity interaction

In [ ]:
interaction_analysis = (
    df_eng.groupby(pd.qcut(df_eng['amount_x_velocity'], q=4, labels=['Q1','Q2','Q3','Q4']))[TARGET]
    .mean().mul(100).round(3)
)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(interaction_analysis.index, interaction_analysis.values, color='steelblue', width=0.6)
ax.set_xlabel('amount × velocity quartile')
ax.set_ylabel('Fraud rate (%)')
ax.set_title('Fraud rate by amount × velocity interaction')
plt.tight_layout()
plt.savefig('../reports/figures/02_interaction_feature.png', bbox_inches='tight')
plt.show()

print(interaction_analysis)

## 4. Validate high_risk_hour

In [ ]:
hourly = df_eng.groupby('transaction_hour')[TARGET].mean().mul(100)

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['tomato' if (h >= 22 or h <= 6) else 'steelblue' for h in hourly.index]
ax.bar(hourly.index, hourly.values, color=colors, width=0.8)
ax.set_xlabel('Transaction hour')
ax.set_ylabel('Fraud rate (%)')
ax.set_title('Fraud rate by hour (red = high-risk hours 22:00–06:00)')
plt.tight_layout()
plt.savefig('../reports/figures/02_high_risk_hour.png', bbox_inches='tight')
plt.show()

## 5. Correlation of new features with target

In [ ]:
all_features = [
    'amount', 'transaction_hour', 'device_trust_score', 'velocity_last_24h', 'cardholder_age',
    'foreign_transaction', 'location_mismatch',
    'amount_log', 'amount_x_velocity', 'high_risk_hour', 'velocity_bin',
    TARGET
]

corr_with_target = (
    df_eng[all_features].corr()[TARGET]
    .drop(TARGET)
    .abs()
    .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(7, 5))
colors = ['tomato' if f in ['amount_log','amount_x_velocity','high_risk_hour','velocity_bin'] else 'steelblue'
          for f in corr_with_target.index]
ax.barh(corr_with_target.index, corr_with_target.values, color=colors)
ax.set_xlabel('|Correlation with is_fraud|')
ax.set_title('Feature correlations with target\n(red = new engineered features)')
plt.tight_layout()
plt.savefig('../reports/figures/02_feature_correlations.png', bbox_inches='tight')
plt.show()

## Feature engineering summary

All four engineered features show a positive marginal association with fraud. `amount_x_velocity` ranks among the strongest predictors overall. These features will be included in the full pipeline via `data_loader.engineer_features()` — no notebook-level transformations needed.